# 09 — Job scheduling across the device pool

This notebook confirms how the benchmark distributes per-model jobs across a pool of devices. The real `GpuQueue` launches each job as a subprocess, binds it to a free device id, and round-robins as devices free up. We drive the genuine queue with trivial sleep jobs (no GPU required) and reconstruct the occupancy timeline.

Modules exercised: `pipelines/shared/orchestration.py` (`GpuQueue`, `GpuJob`, `GpuJobResult`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Defining trivial jobs

Each job is a short Python sleep launched as a subprocess, exactly the launch mechanism the pipeline uses for worker processes. We create more jobs than devices so the queue must schedule in waves.

In [ ]:
import sys
import tempfile
from pathlib import Path

from pipelines.shared.orchestration import GpuQueue, GpuJob
from tools.logger import Logger

work_dir   = Path(tempfile.mkdtemp())
DEVICE_IDS = [0, 1]
DURATIONS  = [0.4, 0.6, 0.3, 0.5, 0.4, 0.3]

jobs = []
for i, seconds in enumerate(DURATIONS):
    jobs.append(GpuJob(
        name     = f"model_{i}",
        command  = [sys.executable, "-c", f"import time; time.sleep({seconds})"],
        log_path = work_dir / f"model_{i}.log",
    ))

print(f"{len(jobs)} jobs over {len(DEVICE_IDS)} devices")
for job, seconds in zip(jobs, DURATIONS):
    print(f"  {job.name}: sleep {seconds}s")

## Running the real queue

`GpuQueue.run` blocks until every job finishes, returning a `GpuJobResult` per job that records which device ran it, its status, and its wall-clock duration. We use a short poll interval so the demonstration completes quickly.

In [ ]:
logger = Logger(log_dir=str(work_dir), name="queue_demo")
queue  = GpuQueue(gpus=DEVICE_IDS, logger=logger, poll_interval_s=0.05)

import time
wall_start = time.monotonic()
results    = queue.run(jobs)
wall_total = time.monotonic() - wall_start
logger.close()

for r in results:
    print(f"{r.name:10s} device {r.gpu}  {r.status:6s}  {r.duration_s:5.2f}s")

print(f"\ntotal wall time: {wall_total:.2f}s")
assert all(r.status == "DONE" for r in results), "a scheduled job failed"
print("all jobs completed")

## Confirming the scheduling invariants

Two properties define correct scheduling: every job ran on exactly one device from the pool, and the queue never used more devices concurrently than the pool size. We check both, then estimate per-device load.

In [ ]:
used_devices = sorted({r.gpu for r in results})
assert set(used_devices).issubset(set(DEVICE_IDS)), "a job ran on an unknown device"

per_device_jobs = {d: [r for r in results if r.gpu == d] for d in DEVICE_IDS}
per_device_load = {d: sum(r.duration_s for r in per_device_jobs[d]) for d in DEVICE_IDS}

for d in DEVICE_IDS:
    names = [r.name for r in per_device_jobs[d]]
    print(f"device {d}: {len(names)} jobs, busy {per_device_load[d]:.2f}s -> {names}")

serial_total = sum(r.duration_s for r in results)
print(f"\nserial total if run one-by-one : {serial_total:.2f}s")
print(f"actual wall time with {len(DEVICE_IDS)} devices : {wall_total:.2f}s")

## Visual confirmation

The left panel is a Gantt-style chart of per-device cumulative busy time, approximating the occupancy timeline. The right panel compares the serial total against the achieved wall time, making the scheduling speed-up visible. With two devices the wall time should be well under the serial total.

In [ ]:
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 4.5))

colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
color_by_name = {r.name: colors[i] for i, r in enumerate(results)}
for d in DEVICE_IDS:
    cursor = 0.0
    for r in per_device_jobs[d]:
        ax_l.barh(d, r.duration_s, left=cursor, color=color_by_name[r.name], edgecolor="white")
        ax_l.text(cursor + r.duration_s / 2, d, r.name, ha="center", va="center", fontsize=7, color="white")
        cursor += r.duration_s
ax_l.set_yticks(DEVICE_IDS)
ax_l.set_yticklabels([f"device {d}" for d in DEVICE_IDS])
ax_l.set_xlabel("cumulative busy time (s)")
ax_l.set_title("Per-device job allocation")

ax_r.bar(["serial", f"{len(DEVICE_IDS)} devices"], [serial_total, wall_total], color=["#a5533b", "#3b6ea5"])
ax_r.set_ylabel("wall time (s)")
ax_r.set_title("Scheduling speed-up")

fig.tight_layout()
plt.show()

## Expected visual outcome

Every job reports `DONE` on a device drawn from the pool, the jobs are spread across both devices, and the actual wall time is markedly less than the serial total. The allocation chart shows both device rows populated and the speed-up chart shows the parallel bar shorter than the serial bar.